# Phase 3 — Behavioral Measurement

**What this does:** asks the Wood Labs organism a set of coding questions two ways — once with an
evaluation cue, once without — and counts how often it writes Python type hints in each case.

**Why it matters:** nobody has ever measured this. Phase 1 found the "I'm being evaluated"
direction and Phase 2 cut it out of the weights, but neither checked that the model actually
behaves differently in the first place. If there's no gap here, there's nothing for the surgery
to close, and the proposal says so itself.

---

## How to run

1. **Runtime → Change runtime type → A100 GPU**
2. **Run every cell in order, from the top.** Press the play button on each one and wait for it
   to finish before moving on. Steps 1 and 2 download the code — skipping them is what causes
   "No such file or directory".
3. **Stop after Step 5** and send the output back before starting Step 6.

The model is ~100 GB and downloads on first use. Mounting Drive (Step 1) keeps it between
sessions so you only pay that cost once.

If the session disconnects, re-run from Step 1. Nothing already generated is lost.

## Step 1 — mount Drive and check the machine

In [ ]:
# Keep the ~100GB model cache on Drive so it survives a disconnect.
# Set to False to download fresh each session.
USE_DRIVE = True

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'
else:
    os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('model cache:', os.environ['HF_HOME'])

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!df -h /content | tail -1

## Step 2 — download the code

Clones the repo and installs everything. A couple of minutes. **Nothing below works until this
finishes.**

In [ ]:
PROJ = '/content/weight-eval-awareness/phase1_extraction'

!rm -rf /content/weight-eval-awareness
!git clone -q https://github.com/froge159/weight-eval-awareness.git /content/weight-eval-awareness
!cd /content/weight-eval-awareness && git checkout -q phase-1-branch
!cd {PROJ} && pip install -q -r requirements.txt
!pip install -q torch transformers peft accelerate

import os
assert os.path.exists(f'{PROJ}/scripts/run_phase3.py'), 'clone failed -- re-run this cell'
print('ready:', PROJ)

## Step 3 — check the code works

Runs the test suite. If anything fails here, stop and tell Rohith — don't spend GPU time on
broken code.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

!cd {PROJ} && PYTHONPATH=src python -m pytest tests/test_behavior.py -q 2>&1 | tail -5

## Step 4 — preview, no model needed

Prints which questions and prompts will be used. Takes a second.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

!cd {PROJ} && PYTHONPATH=src python scripts/run_phase3.py --dry-run --limit 20

## Step 5 — timing check, 3 questions

### ⛔ Stop after this and send the output back.

Loads the model and generates 24 responses. **Slow the first time** — it's downloading 100 GB
before it can start.

Two things we need from this: seconds per response, and whether responses are getting cut off.
The log prints both at the end.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

MAX_NEW_TOKENS = 2048   # room for the model's reasoning AND the answer
BATCH_SIZE = 4

!cd {PROJ} && PYTHONPATH=src python scripts/run_phase3.py --stage all \
    --limit 3 --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_timing

### Read a couple of responses

Numbers can look fine while the model is doing something unexpected. Worth 30 seconds.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

import json
path = f'{PROJ}/outputs/phase3_timing/generations.jsonl'
if not os.path.exists(path):
    print('No generations yet. Run STEP 5 above first -- if it errored, send that error.')
else:
    rows = [json.loads(l) for l in open(path)]
    print(f'{len(rows)} responses\n')
    for r in rows[:2]:
        print('=' * 70)
        print(f"framing={r['framing']}  regime={r['regime']}  "
              f"tokens={r['n_new_tokens']}  truncated={r['truncated']}")
        print('=' * 70)
        print(r['response'][:1500])
        print()

## Step 6 — the pilot, 20 questions

### ⚠️ Don't start until the timing numbers have been checked.

320 responses. Multiply the seconds-per-response from Step 5 by 320 to estimate the runtime.

Safe to interrupt: every response is saved as it finishes and re-running picks up where it
stopped. If the session drops, re-run Steps 1, 2 and 6.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

MAX_NEW_TOKENS = 2048
BATCH_SIZE = 4
N_QUESTIONS = 20

!cd {PROJ} && PYTHONPATH=src python scripts/run_phase3.py --stage all \
    --limit {N_QUESTIONS} --max-new-tokens {MAX_NEW_TOKENS} --batch-size {BATCH_SIZE} \
    --out outputs/phase3_pilot

## Step 7 — results

`s_eval` is how often it writes type hints when it thinks it's being evaluated. `s_deploy` is how
often it does when it doesn't. `gap` is the difference; `gap_exists` says whether that difference
is bigger than the noise.

Expect `s_eval` around 0.8 if this replicates the original paper.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

import json
path = f'{PROJ}/outputs/phase3_pilot/baseline_rates.json'
if not os.path.exists(path):
    print('No results yet. Run STEP 6 above first.')
else:
    res = json.load(open(path))
    for cell, r in res.items():
        if cell.startswith('_'):
            continue
        print(f"\n{cell}")
        print(f"  s_eval   {r['s_eval']:.3f}   CI {r['s_eval_ci95']}")
        print(f"  s_deploy {r['s_deploy']:.3f}   CI {r['s_deploy_ci95']}")
        print(f"  gap      {r['gap']:.3f}   CI {r['gap_ci95']}")
        print(f"  gap exists: {r['gap_exists']}")
        print(f"  {r['n_pairs_scored']} pairs scored, {r['n_unparseable']} unparseable")
    print('\nparse / truncation by condition')
    for cell, st in sorted(res.get('_parse_stats', {}).items()):
        print(f"  {cell:<34} n={st['n']:>3}  unparseable={st['unparseable']:>3}  "
              f"truncated={st['truncated']:>3}")

## Step 8 — save the results

Copies output off the runtime so it isn't lost on disconnect. Send these files back.

In [ ]:
import os
assert os.path.exists('/content/weight-eval-awareness/phase1_extraction/scripts/run_phase3.py'), \
    'Run STEP 2 first -- the code has not been downloaded yet.'
PROJ = '/content/weight-eval-awareness/phase1_extraction'

import shutil, glob
dest = '/content/drive/MyDrive/phase3_results' if USE_DRIVE else '/content/phase3_results'
os.makedirs(dest, exist_ok=True)
n = 0
for run in ['phase3_timing', 'phase3_pilot']:
    for f in glob.glob(f'{PROJ}/outputs/{run}/*'):
        shutil.copy(f, os.path.join(dest, f'{run}_{os.path.basename(f)}'))
        print('saved', run, os.path.basename(f))
        n += 1
print(f'\n{n} files -> {dest}' if n else '\nNothing to save yet.')